###### Importing required libraries 

In [1]:
from utils import Load_Rumours_Dataset_filtering_since_first_post
import numpy as np
import pandas as pd
from sklearn.metrics import *
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings("ignore")
event_name ="charlie_hebdo"

In [2]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


#### Testing a single load 

In [13]:
processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=1266)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [14]:
test.shape

(324, 10)

In [4]:
previous_node_count = 0

In [5]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])

X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])


y_test =test['rumour']
y_test_new = test.iloc[previous_node_count:]['rumour']

previous_node_count = test.shape[0]
print(f"New Instances: {X_test_new.shape[0]}")

New Instances: 352


In [3]:




# Model definition
class RumorDetectionLSTM(nn.Module):


    """
    Hybrid neural network model for rumor detection using embeddings and structured features.

    This model combines an LSTM network that processes precomputed text embeddings
    with fully connected layers that process additional handcrafted or metadata features.
    The outputs from these two branches are fused and passed through dense layers
    for binary rumor classification.

    Architecture Overview:
    ---------------------
    • Input features are divided into:
        - First 8 features: handcrafted / numerical indicators
        - Last 100 features: embedding vector representing textual content
    • A single-step LSTM processes the embedding to extract contextual semantics
    • Dense layers extract nonlinear structure from auxiliary features
    • Concatenated representation is classified with fully connected layers

    Parameters
    ----------
    embedding_dim : int, default=100
        Dimensionality of the input embedding vector.
    lstm_hidden_size : int, default=32
        Number of hidden units in the LSTM layer.
    dense_hidden_size : int, default=16
        Number of hidden units in the dense feature branch.

    Forward Input
    -------------
    x : torch.Tensor
        Tensor of shape (batch_size, 108) where:
        - x[:, :8] are handcrafted features
        - x[:, -100:] is a sentence/document embedding

    Returns
    -------
    torch.Tensor
        A 1D tensor of shape (batch_size,) containing probabilities in [0, 1]
        where values close to 1 indicate high likelihood of rumor.

    Notes
    -----
    - Assumes sequence length = 1 in embedding branch.
    - Uses sigmoid activation for binary classification tasks.
    """
    
    def __init__(self, embedding_dim=100, lstm_hidden_size=32, dense_hidden_size=16):
        super(RumorDetectionLSTM, self).__init__()
        
        # LSTM for the 100-dimensional embeddings
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=lstm_hidden_size, batch_first=True)
        
        # Dense layers for other features
        self.dense1 = nn.Linear(8, 16)  # 8 non-embedding features
        self.dense2 = nn.Linear(16, dense_hidden_size)
        
        # Combine LSTM and dense features
        self.fc1 = nn.Linear(lstm_hidden_size + dense_hidden_size, 64)
        self.fc2 = nn.Linear(64, 1)
        
    def forward(self, x):
        # Separate embeddings and other features
        embeddings = x[:, -100:].unsqueeze(1)  # (batch, seq_len=1, embedding_dim)
        other_features = x[:, :8]  # First 8 features
        
        # LSTM output
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = lstm_out[:, -1, :]  # Get the last LSTM output
        
        # Dense layers for other features
        dense_out = torch.relu(self.dense1(other_features))
        dense_out = torch.relu(self.dense2(dense_out))
        
        # Concatenate LSTM and dense outputs
        combined = torch.cat((lstm_out, dense_out), dim=1)
        
        # Fully connected layers for classification
        x = torch.relu(self.fc1(combined))
        x = torch.sigmoid(self.fc2(x))
        return x.squeeze()


#### Example  training

In [4]:
# Assuming X_train, X_test, y_train, and y_test are available as numpy arrays
# Convert them to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# Dataset and DataLoader
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

NameError: name 'X_train' is not defined

In [8]:


# Model, criterion, optimizer initialization (as before)
model = RumorDetectionLSTM()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop with loss and recall monitoring
epochs = 75  # Adjust as needed
train_recall_interval = 50  # Calculate train recall every 10 epochs
loss_interval = 50  # Print loss every 10 epochs

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # Print loss every 10 epochs
    if (epoch + 1) % loss_interval == 0:
        model.eval()
        train_preds = []
        train_labels = []
        with torch.no_grad():
            for X_batch, y_batch in train_loader:
                output = model(X_batch)
                preds = (output >= 0.5).int()  # Binarize predictions
                train_preds.extend(preds.tolist())
                train_labels.extend(y_batch.tolist())
        
        train_recall = recall_score(train_labels, train_preds)
        train_precision = precision_score(train_labels, train_preds)
        
print(f"Epoch {epoch + 1}, Train Loss: {epoch_loss / len(train_loader):.4f},\
              Train Precision: {train_precision:.4f},Train Recall: {train_recall:.4f}")
    


# Final evaluation on test set with recall and precision
model.eval()
test_preds = []
test_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds = (output >= 0.5).int()  # Binarize predictions
        test_preds.extend(preds.tolist())
        test_labels.extend(y_batch.tolist())

# Calculate final test recall and precision
test_recall = recall_score(test_labels, test_preds)
test_precision = precision_score(test_labels, test_preds)

print(f"Final Test Recall: {test_recall:.4f}")
print(f"Final Test Precision: {test_precision:.4f}")


Epoch 75, Train Loss: 0.0297,              Train Precision: 0.9834,Train Recall: 0.9308
Final Test Recall: 0.8705
Final Test Precision: 0.7246


In [9]:
# Model, criterion, optimizer initialization (as before)
model = RumorDetectionLSTM()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 200
loss_interval = 50

# Save train outputs for threshold tuning
train_probs_all = []
train_labels_all = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    # Store train outputs for final threshold selection
    if (epoch + 1) == epochs:
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in train_loader:
                output = model(X_batch)
                train_probs_all.extend(output.squeeze().tolist())
                train_labels_all.extend(y_batch.squeeze().tolist())
    
    # Logging
    if (epoch + 1) % loss_interval == 0:
        print(f"Epoch {epoch + 1}, Train Loss: {epoch_loss / len(train_loader):.4f}")

# ------------------------
# Find best threshold on train set
# ------------------------
train_probs_all = np.array(train_probs_all)
train_labels_all = np.array(train_labels_all)

best_thresh = 0.5
best_f1 = 0.0
for t in np.arange(0.0, 1.01, 0.01):
    preds = (train_probs_all >= t).astype(int)
    f1 = f1_score(train_labels_all, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"\nBest Threshold on Train (max F1): {best_thresh:.2f} | F1: {best_f1:.4f}")

# ------------------------
# Final evaluation on test set using best threshold
# ------------------------
model.eval()
test_probs = []
test_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        test_probs.extend(output.squeeze().tolist())
        test_labels.extend(y_batch.squeeze().tolist())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds = (test_probs >= best_thresh).astype(int)

# Calculate final metrics
test_recall = recall_score(test_labels, test_preds)
test_precision = precision_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds)

print(f"Final Test Recall: {test_recall:.4f}")
print(f"Final Test Precision: {test_precision:.4f}")
print(f"Final Test F1: {test_f1:.4f}")


Epoch 50, Train Loss: 0.0981
Epoch 100, Train Loss: 0.0066
Epoch 150, Train Loss: 0.0011
Epoch 200, Train Loss: 0.0003

Best Threshold on Train (max F1): 0.02 | F1: 1.0000
Final Test Recall: 0.9424
Final Test Precision: 0.6823
Final Test F1: 0.7915


#### Setting MLflow Experiment

In [ ]:
from datetime import date
import mlflow

today = date.today()
formatted_today = today.strftime("%Y-%m-%d")


mlflow.set_experiment(f"LSTM {formatted_today} {event_name}")

#### Loading dataset statistics to get the final time cut 

In [ ]:
import pandas as pd
df_posts_by_time_cut = pd.read_pickle(f'replies_{event_name}.pkl')

df_posts_by_time_cut['min_since_fst_post'] = round(
            (df_posts_by_time_cut['time'] - df_posts_by_time_cut['time'].min()).dt.total_seconds() / 60, 2)


In [ ]:
df_metrics = df_posts_by_time_cut[['id','time','rumour','min_since_fst_post']].drop_duplicates().sort_values(by='time')

In [ ]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=10000)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [ ]:

start = test.min_since_fst_post.min()
end = test.min_since_fst_post.max()
duration = end-start
experiment_time= duration+60



In [ ]:
print('start: ',start)
print('end: ',end)
print('experiment_time: ',experiment_time)

In [ ]:
def compute_metrics_custom(df, prob_col='prob', target_col='rumour'):
    df = df.copy()
    df = df.sort_values(prob_col, ascending=False).reset_index(drop=True)

    total_frauds = df[target_col].sum()
    total_records = df.shape[0]
    n = len(df)

    # ✅ Bins from 1% to 100% in 1% steps
    bins = np.arange(0.01, 1.01, 0.01)

    results = []

    for p in bins:
        cutoff = int(np.ceil(n * p))
        subset = df.iloc[:cutoff]

        frauds = subset[target_col].sum()
        records = len(subset)

        results.append({
            'percentile': round(p * 100, 0),
            'records': records,
            'frauds_captured': frauds,
            'capture_rate': frauds / total_frauds if total_frauds > 0 else 0,
            'bad_rate': frauds / records if records > 0 else 0,
            'false_positive_rate': (records - frauds) / total_records if records > 0 else 0
        })

    df_out = (
        pd.DataFrame(results)
        .drop_duplicates(subset='percentile')
        .sort_values('percentile')
        .reset_index(drop=True)
    )

    return df_out

In [ ]:
new_posts_times = np.sort(
    np.unique(
        np.ceil(
            df_metrics[
                (df_metrics.min_since_fst_post >= start ) &
                (df_metrics.min_since_fst_post <= end)
            ].min_since_fst_post - start
        )
    )
)

In [ ]:
new_posts_times=[c for c in new_posts_times if c >10]

In [ ]:
previous_node_count =324

for time_cut in np.linspace(0, int(experiment_time), 50):
    time_cut=int(time_cut)
    if time_cut<=7:
        continue
    print(f"\n=== Time Cut: {time_cut} minutes ===")

    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train,test= processor.get_final_dataframes()


    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    print(f"New Instances: {X_test_new.shape[0]}")


    # Handle class imbalance
    num_pos = sum(y_train)
    num_neg = len(y_train) - num_pos
    pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32)

    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train.values, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    X_test_new = torch.tensor(X_test_new, dtype=torch.float32)
    y_test = torch.tensor(y_test.values, dtype=torch.float32)
    y_test_new = torch.tensor(y_test_new.values, dtype=torch.float32)

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    test_dataset_new= TensorDataset(X_test_new, y_test_new)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)
    test_loader_new = DataLoader(test_dataset_new, batch_size=32)

    # Define model, loss, and optimizer
    model = RumorDetectionLSTM()  # <- define your model class elsewhere
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    epochs = 200
    loss_interval = 50

    train_probs_all = []
    train_labels_all = []

    with mlflow.start_run():
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                output = model(X_batch).view(-1)
                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            if (epoch + 1) % loss_interval == 0:
                print(f"Epoch {epoch + 1}, Train Loss: {epoch_loss / len(train_loader):.4f}")

            # Store outputs for threshold tuning after last epoch
            if (epoch + 1) == epochs:
                model.eval()
                with torch.no_grad():
                    for X_batch, y_batch in train_loader:
                        output = model(X_batch).view(-1)
                        train_probs_all.extend(output.tolist())
                        train_labels_all.extend(y_batch.tolist())

        # Threshold tuning (maximize F1)
        train_probs_all = np.array(train_probs_all)
        train_labels_all = np.array(train_labels_all)
        #best_thresh = 0.5
        #best_f1 = 0.0
        #for t in np.arange(0.0, 1.01, 0.01):
        #    preds = (train_probs_all >= t).astype(int)
        #    f1 = f1_score(train_labels_all, preds)
        #    if f1 > best_f1:
        #        best_f1 = f1
        #        best_thresh = t

        #print(f"\nBest Threshold on Train (max F1): {best_thresh:.2f} | F1: {best_f1:.4f}")

        # Final test evaluation
        model.eval()
        test_probs = []
        test_labels = []

        test_probs_new = []
        test_labels_new = []
        
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                output = model(X_batch).view(-1)
                test_probs.extend(output.tolist())
                test_labels.extend(y_batch.tolist())

                
      
            

        test_probs = np.array(test_probs)
        #test_labels = np.array(test_labels)
        #test_preds = (test_probs >= best_thresh).astype(int)
        current_df_metrics = df_metrics.iloc[X_train.shape[0]:X_train.shape[0]+X_test.shape[0]]
        current_df_metrics['prob'] = test_probs
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)

        df_metrics_by_bucket.to_csv(f"metrics_by_bucket_{event_name}_LSTM.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_{event_name}_LSTM.csv")
 
     

 
        mlflow.log_param("learning_rate", 0.001)
        mlflow.log_param("epochs", epochs)
        mlflow.log_metric("new_posts", X_test_new.shape[0])
        mlflow.log_metric("time_cut", time_cut)


In [ ]:
previous_node_count = 0
for time_cut in new_posts_times:
    print(f"\n=== Time Cut: {time_cut} minutes ===")

    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train,test= processor.get_final_dataframes()


    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    print(f"New Instances: {X_test_new.shape[0]}")


    # Handle class imbalance
    num_pos = sum(y_train)
    num_neg = len(y_train) - num_pos
    pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32)

    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train.values, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    X_test_new = torch.tensor(X_test_new, dtype=torch.float32)
    y_test = torch.tensor(y_test.values, dtype=torch.float32)
    y_test_new = torch.tensor(y_test_new.values, dtype=torch.float32)

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    test_dataset_new= TensorDataset(X_test_new, y_test_new)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32)
    test_loader_new = DataLoader(test_dataset_new, batch_size=32)

    # Define model, loss, and optimizer
    model = RumorDetectionLSTM()  # <- define your model class elsewhere
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    epochs = 200
    loss_interval = 50

    train_probs_all = []
    train_labels_all = []

    with mlflow.start_run():
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                output = model(X_batch).view(-1)
                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            if (epoch + 1) % loss_interval == 0:
                print(f"Epoch {epoch + 1}, Train Loss: {epoch_loss / len(train_loader):.4f}")

            # Store outputs for threshold tuning after last epoch
            if (epoch + 1) == epochs:
                model.eval()
                with torch.no_grad():
                    for X_batch, y_batch in train_loader:
                        output = model(X_batch).view(-1)
                        train_probs_all.extend(output.tolist())
                        train_labels_all.extend(y_batch.tolist())

        # Threshold tuning (maximize F1)
        train_probs_all = np.array(train_probs_all)
        train_labels_all = np.array(train_labels_all)
        #best_thresh = 0.5
        #best_f1 = 0.0
        #for t in np.arange(0.0, 1.01, 0.01):
        #    preds = (train_probs_all >= t).astype(int)
        #    f1 = f1_score(train_labels_all, preds)
        #    if f1 > best_f1:
        #        best_f1 = f1
        #        best_thresh = t

        #print(f"\nBest Threshold on Train (max F1): {best_thresh:.2f} | F1: {best_f1:.4f}")

        # Final test evaluation
        model.eval()
        test_probs = []
        test_labels = []

        test_probs_new = []
        test_labels_new = []
        
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                output = model(X_batch).view(-1)
                test_probs.extend(output.tolist())
                test_labels.extend(y_batch.tolist())

                
      
            

        test_probs = np.array(test_probs)
        #test_labels = np.array(test_labels)
        #test_preds = (test_probs >= best_thresh).astype(int)
        current_df_metrics = df_metrics.iloc[X_train.shape[0]:X_train.shape[0]+X_test.shape[0]]
        current_df_metrics['prob'] = test_probs
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)

        if X_test_new.shape[0] >0:
            #y_test_new_prob = model.predict_proba(X_test_new)[:, 1]
            new_df_metrics = current_df_metrics.iloc[-X_test_new.shape[0]:]
            #new_df_metrics['prob'] = y_test_new_prob
            new_df_metrics_by_bucket = compute_metrics_custom(new_df_metrics)

            new_df_metrics_by_bucket.to_csv(f"metrics_by_bucket_new_posts_{event_name}_LSTM.csv", index=False)
            mlflow.log_artifact(f"metrics_by_bucket_new_posts_{event_name}_LSTM.csv")
        else: 
            continue
 
     

 
        mlflow.log_param("learning_rate", 0.001)
        mlflow.log_param("epochs", epochs)
        mlflow.log_metric("new_posts", X_test_new.shape[0])
        mlflow.log_metric("time_cut", time_cut)
